<a href="https://colab.research.google.com/github/zly554411-arch/ECON3916-Statistical-Machine-Learning/blob/main/Lab%2012/%5BLab_12_%5D_OLS%2C_Hedonic_Pricing%2C_and_RMSE_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

# Step 1: Ingestion from external source
url = 'https://raw.githubusercontent.com/zly554411-arch/ECON3916-Statistical-Machine-Learning/refs/heads/main/Zillow_ZHVI_2026_Micro.csv'
df = pd.read_csv(url)

df.head()

,Home_Value,Square_Footage,Property_Age,Distance_to_Transit,School_District_Rating
0,329705.74,1941.0,5.5,6.45,Excellent
1,183343.63,1364.3,35.2,2.15,Average
2,354551.73,2386.9,52.4,0.75,Good
3,325773.17,2192.1,50.2,5.25,Excellent
4,359743.12,3069.8,66.5,12.69,Excellent


In [2]:
# Step 2: Defining the formula
# Utilizing the R-style patsy formula interface allows for elegant, readable model specification
formula ='Home_Value ~ Square_Footage	+ Property_Age	+ Distance_to_Transit	+ School_District_Rating'

In [3]:
# Step 3: Fitting the model and printing the summary
model = smf.ols(formula=formula, data=df)
results = model.fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     542.5
Date:                Mon, 16 Mar 2026   Prob (F-statistic):          2.81e-309
Time:                        19:57:49   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.416e+04
Df Residuals:                     993   BIC:                         2.419e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [5]:
# Step 4: Generating predictions
# We extract the predicted values vector to transition from explanation to prediction
y_pred = results.predict(df)

y_pred

,0
0,312288.586866
1,223813.440910
2,331610.316284
3,307402.426806
4,392714.851722
...,...
995,406437.277604
996,380041.214086
997,236618.275309
998,315508.480776


In [6]:
# Step 5: Calculate RMSE between the actuals and the predictions
model_rmse = rmse(df["Home_Value"], y_pred)
print(f"\nThe Predictive RMSE is: ${model_rmse:,.2f}")


The Predictive RMSE is: $42,316.69


In [7]:
# =============================================================================
# RESIDUAL FORENSICS DASHBOARD
# Hedonic Pricing OLS Model — Interactive Diagnostic Visualisation
# Requires: statsmodels, plotly, pandas, numpy, scikit-learn
# =============================================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go
from sklearn.metrics import mean_squared_error

# =============================================================================
# SECTION 1 — SYNTHETIC DATA (replace with your own DataFrame + OLS results)
# =============================================================================
# This block simulates a hedonic pricing dataset (e.g. property prices).
# Swap this out for your actual `results` object from statsmodels.

np.random.seed(42)
n = 300

# Simulate hedonic features: size (sqft), age (years), rooms, distance to CBD (km)
X_raw = pd.DataFrame({
    "sqft":     np.random.normal(1500, 400, n),
    "age":      np.random.uniform(0, 40, n),
    "rooms":    np.random.randint(1, 6, n).astype(float),
    "dist_cbd": np.random.uniform(1, 30, n),
})

# True price relationship with heteroscedastic noise (variance grows with sqft)
noise = np.random.normal(0, 40_000 + X_raw["sqft"] * 30, n)
y = (
    200_000
    + X_raw["sqft"]    * 120
    - X_raw["age"]     * 500
    + X_raw["rooms"]   * 8_000
    - X_raw["dist_cbd"]* 3_000
    + noise
)

# Add statsmodels constant column for the intercept term
X = sm.add_constant(X_raw)

# Fit OLS — this is the statsmodels RegressionResults object your lab produced
results = sm.OLS(y, X).fit()

# =============================================================================
# SECTION 2 — EXTRACT RESIDUALS & FITTED VALUES FROM statsmodels
# =============================================================================

# results.fittedvalues → pandas Series of ŷ (predicted values) for each obs.
# Internally statsmodels computes ŷ = X @ β̂ ; this property exposes that array.
fitted = results.fittedvalues

# results.resid → pandas Series of raw OLS residuals: e = y − ŷ
# These are NOT standardised; they preserve the original response-variable scale.
residuals = results.resid

# =============================================================================
# SECTION 3 — OUTLIER DETECTION (±2σ threshold)
# =============================================================================

# Compute the standard deviation of the residual distribution.
# We use the full-sample std (ddof=1) of raw residuals as our dispersion measure.
resid_std = residuals.std()

# Boolean mask: True where |eᵢ| > 2σ  →  these are our flagged outliers.
# The 2σ rule captures the ~5% most extreme residuals under normality.
is_outlier = residuals.abs() > 2 * resid_std

# Separate the two populations for distinct visual treatment.
df_normal  = pd.DataFrame({"fitted": fitted[~is_outlier], "resid": residuals[~is_outlier]})
df_outlier = pd.DataFrame({"fitted": fitted[ is_outlier], "resid": residuals[ is_outlier]})

# =============================================================================
# SECTION 4 — RMSE (matches what your lab computed)
# =============================================================================

# mean_squared_error with squared=False is deprecated in newer sklearn.
# Use np.sqrt(mean_squared_error(...)) for forward-compatibility.
rmse = np.sqrt(mean_squared_error(y, fitted))

# =============================================================================
# SECTION 5 — LOESS SMOOTH TREND LINE (manual rolling — no deprecated calls)
# =============================================================================
# A local-linear smoother through the residuals reveals systematic curvature
# that OLS's global linearity assumption may have missed.

sort_idx   = fitted.argsort()
fitted_s   = fitted.iloc[sort_idx].values
resid_s    = residuals.iloc[sort_idx].values

window     = max(20, n // 15)               # adaptive window ~6–7% of data
half_w     = window // 2
smooth_y   = np.full(n, np.nan)

for i in range(n):
    lo = max(0, i - half_w)
    hi = min(n, i + half_w)
    smooth_y[i] = np.mean(resid_s[lo:hi])  # local mean approximates LOESS

# =============================================================================
# SECTION 6 — BUILD THE PLOTLY FIGURE
# =============================================================================

fig = go.Figure()

# ── Normal residuals ──────────────────────────────────────────────────────────
# plotly.graph_objects.Scatter avoids any deprecated px helpers.
fig.add_trace(go.Scatter(
    x         = df_normal["fitted"],
    y         = df_normal["resid"],
    mode      = "markers",
    name      = "Residual",
    marker    = dict(
        color   = "rgba(99, 160, 210, 0.55)",   # steel blue, semi-transparent
        size    = 6,
        line    = dict(width=0.4, color="rgba(60,120,180,0.6)"),
    ),
    hovertemplate = (
        "<b>Fitted:</b> $%{x:,.0f}<br>"
        "<b>Residual:</b> $%{y:,.0f}<extra></extra>"
    ),
))

# ── Outlier residuals (> 2σ) — stark crimson ─────────────────────────────────
fig.add_trace(go.Scatter(
    x         = df_outlier["fitted"],
    y         = df_outlier["resid"],
    mode      = "markers",
    name      = "Outlier (|e| > 2σ)",
    marker    = dict(
        color  = "#DC143C",                     # CSS Crimson — high-contrast flag
        size   = 9,
        symbol = "circle-open",                 # hollow ring for visual salience
        line   = dict(width=2, color="#DC143C"),
    ),
    hovertemplate = (
        "<b>⚠ OUTLIER</b><br>"
        "<b>Fitted:</b> $%{x:,.0f}<br>"
        "<b>Residual:</b> $%{y:,.0f}<extra></extra>"
    ),
))

# ── LOESS trend ───────────────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x         = fitted_s,
    y         = smooth_y,
    mode      = "lines",
    name      = "Local Mean Trend",
    line      = dict(color="#F4A300", width=2.2, dash="dot"),  # amber dashed
    hoverinfo = "skip",
))

# ── Zero reference line (ŷ − y = 0) ──────────────────────────────────────────
# add_hline is the current, non-deprecated API for horizontal shapes.
fig.add_hline(
    y           = 0,
    line_width  = 1.5,
    line_dash   = "dash",
    line_color  = "rgba(255,255,255,0.55)",
    annotation_text      = "Zero Error",
    annotation_position  = "bottom right",
    annotation_font      = dict(color="rgba(255,255,255,0.5)", size=11),
)

# ── ±2σ band ──────────────────────────────────────────────────────────────────
for sign, label in [(1, "+2σ"), (-1, "−2σ")]:
    fig.add_hline(
        y           = sign * 2 * resid_std,
        line_width  = 1,
        line_dash   = "longdash",
        line_color  = "rgba(220,20,60,0.45)",
        annotation_text     = label,
        annotation_position = "top right" if sign == 1 else "bottom right",
        annotation_font     = dict(color="rgba(220,20,60,0.7)", size=11),
    )

# ── Layout ────────────────────────────────────────────────────────────────────
fig.update_layout(
    title = dict(
        text     = (
            "Residual Forensics Dashboard — Hedonic OLS<br>"
            f"<sup>RMSE: ${rmse:,.0f} &nbsp;|&nbsp; "
            f"Outliers flagged: {is_outlier.sum()} / {n} obs &nbsp;|&nbsp; "
            f"R²: {results.rsquared:.4f}</sup>"
        ),
        font     = dict(size=17, color="#E8E8E8"),
        x        = 0.5,
        xanchor  = "center",
    ),
    xaxis = dict(
        title      = dict(text="Fitted Values (ŷ)  [ $ ]", font=dict(size=13)),
        tickformat = "$,.0f",
        gridcolor  = "rgba(255,255,255,0.06)",
        zeroline   = False,
    ),
    yaxis = dict(
        title      = dict(text="Residuals  (y − ŷ)  [ $ ]", font=dict(size=13)),
        tickformat = "$,.0f",
        gridcolor  = "rgba(255,255,255,0.06)",
        zeroline   = False,
    ),
    legend = dict(
        bgcolor     = "rgba(30,30,30,0.7)",
        bordercolor = "rgba(255,255,255,0.15)",
        borderwidth = 1,
        font        = dict(color="#CCCCCC"),
    ),
    plot_bgcolor  = "#0F1117",
    paper_bgcolor = "#0F1117",
    font          = dict(color="#CCCCCC", family="Courier New, monospace"),
    margin        = dict(l=70, r=50, t=110, b=70),
    height        = 600,
    hovermode     = "closest",
)

fig.show()

# Optional: save as self-contained HTML
fig.write_html("residual_forensics_dashboard.html")
print(f"\n✓  Dashboard saved → residual_forensics_dashboard.html")
print(f"   RMSE            : ${rmse:,.2f}")
print(f"   Outliers (>2σ)  : {is_outlier.sum()} of {n} observations")
print(f"   R²              : {results.rsquared:.4f}")
print(f"   Adj. R²         : {results.rsquared_adj:.4f}")


✓  Dashboard saved → residual_forensics_dashboard.html
   RMSE            : $92,911.79
   Outliers (>2σ)  : 17 of 300 observations
   R²              : 0.2973
   Adj. R²         : 0.2878
